
# Lantern Second Pass on hybrid ANTARES data — multi-loci version

This notebook (previously named `lantern_second_pass_hybrid_multi_loci_v3`) applies the Lantern Second-Pass model to the downloaded hybrid ANTARES dataset.

Workflow:

1. read loci/alerts from the `antares_data_hybrid/` Parquet directories;
2. keep the Lantern operating period (`MJD >= 61187.0`) and alerts containing `lsst_diaSource_band`;
3. keep First-Cut loci with `max_score > 0.9691`;
4. group nearby loci with a **3 arcsec transitive friends-of-friends (FoF)** grouping;
5. combine all retained alerts from all member loci into one target;
6. construct the same **44 Second-Pass model features** as the earlier local notebook;
7. score every target with the Second-Pass model;
8. select candidates with `second_pass_score > 0.80`;
9. save target membership, scores, candidates, and candidate alerts.

## ANTARES Lantern counter mismatch

We found a small number of loci for which ANTARES reports, for example,

```text
num_tagged_alerts = 3
unique alerts available from ANTARES = 2
```

These targets are **kept**.

For every grouped target:

- `n_tagged` keeps the Lantern count. For a multi-locus target, it is the **sum** of the member-locus Lantern counts.
- `bad_counter_flag` is kept only as a diagnostic flag.
- The original Second-Pass model defines the tagging percentage from `n_detections`.
- We cap this feature at 100%:

```python
percent_tagged = min(100 * n_tagged / n_detections, 100)
```

So a counter mismatch can never give the model `percent_tagged > 100`.


In [ ]:

from pathlib import Path
import os
import ast
import json
import pickle
from collections.abc import Mapping

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree


# ============================================================
# INPUTS
# ============================================================

# Adjust this path if the notebook is stored somewhere else.
DATA_DIR = Path("../download_v4/antares_data_hybrid")

LOCI_DIR = DATA_DIR / "loci"
ALERTS_DIR = DATA_DIR / "alerts"

MODEL_FILE = "second_pass_first_test.pkl"

# Lantern First Cut
FIRST_CUT_THRESHOLD = 0.9691

# Lantern operating period
FILTER_START_MJD = 61187.0

# Multi-locus grouping radius
FOF_RADIUS_ARCSEC = 3.0

# Run all grouped targets.
# Set <1 only for a quick test.
SAMPLE_FRAC = 1.0
RANDOM_STATE = 42

# Second-Pass candidate threshold
SECOND_PASS_THRESHOLD = 0.80

OUTPUT_DIR = Path("second_pass_output_hybrid_multi_loci")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load hybrid Parquet data and model

In [ ]:

loci_columns = [
    "locus_id",
    "ra",
    "dec",
    "max_score",
    "num_tagged_alerts",
]

alert_columns = [
    "locus_id",
    "alert_id",
    "mjd",
    "alert_properties",
]


loci = (
    ds.dataset(
        LOCI_DIR,
        format="parquet",
    )
    .to_table(columns=loci_columns)
    .to_pandas()
)

alerts = (
    ds.dataset(
        ALERTS_DIR,
        format="parquet",
    )
    .to_table(columns=alert_columns)
    .to_pandas()
)


with open(MODEL_FILE, "rb") as f:
    model = pickle.load(f)


required_locus = set(loci_columns)
required_alert = set(alert_columns)

missing = required_locus - set(loci.columns)
if missing:
    raise KeyError(
        f"Missing locus columns: {sorted(missing)}"
    )

missing = required_alert - set(alerts.columns)
if missing:
    raise KeyError(
        f"Missing alert columns: {sorted(missing)}"
    )


print(f"Loci:                 {len(loci):,}")
print(f"Unique locus IDs:      {loci['locus_id'].nunique():,}")
print(f"Alerts:                {len(alerts):,}")
print(f"Unique alert IDs:      {alerts['alert_id'].nunique():,}")
print(f"Model:                 {MODEL_FILE}")


# The clean hybrid download is expected to be globally unique.
if loci["locus_id"].duplicated().any():
    raise ValueError(
        "Duplicate locus_id rows found in the input loci dataset."
    )

if alerts["alert_id"].duplicated().any():
    raise ValueError(
        "Duplicate alert_id rows found in the input alert dataset."
    )

if loci[["ra", "dec"]].isna().any().any():
    raise ValueError(
        "Some loci have missing RA/Dec; FoF grouping requires coordinates."
    )


## 2. Helpers

In [ ]:

def as_dict(value):
    if isinstance(value, Mapping):
        return value

    if isinstance(value, str):
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(value)
                if isinstance(parsed, Mapping):
                    return parsed
            except Exception:
                pass

    return {}


def get_prop(value, key, default=np.nan):
    value = as_dict(value)
    v = value.get(key, default)
    return default if v is None else v


def has_prop(value, key):
    return key in as_dict(value)


def normalize_band(series):
    s = series.astype(str)

    extracted = s.str.extract(
        r"band=['\"]([ugrizy])['\"]",
        expand=False,
    )

    plain = s.str.extract(
        r"^([ugrizy])$",
        expand=False,
    )

    return extracted.fillna(plain)


def radec_to_unit_xyz(ra_deg, dec_deg):
    ra = np.deg2rad(
        np.asarray(ra_deg, dtype=float)
    )
    dec = np.deg2rad(
        np.asarray(dec_deg, dtype=float)
    )

    cosdec = np.cos(dec)

    return np.column_stack([
        cosdec * np.cos(ra),
        cosdec * np.sin(ra),
        np.sin(dec),
    ])


def spherical_mean_deg(ra_deg, dec_deg):
    xyz = radec_to_unit_xyz(
        ra_deg,
        dec_deg,
    )

    mean_xyz = np.nanmean(
        xyz,
        axis=0,
    )

    norm = np.linalg.norm(mean_xyz)

    if not np.isfinite(norm) or norm == 0:
        return np.nan, np.nan

    mean_xyz = mean_xyz / norm

    ra = np.rad2deg(
        np.arctan2(
            mean_xyz[1],
            mean_xyz[0],
        )
    ) % 360.0

    dec = np.rad2deg(
        np.arcsin(
            np.clip(
                mean_xyz[2],
                -1.0,
                1.0,
            )
        )
    )

    return ra, dec


class UnionFind:
    def __init__(self, n):
        self.parent = np.arange(n)
        self.rank = np.zeros(
            n,
            dtype=np.int8,
        )

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[
                self.parent[x]
            ]
            x = self.parent[x]

        return x

    def union(self, a, b):
        ra = self.find(a)
        rb = self.find(b)

        if ra == rb:
            return

        if self.rank[ra] < self.rank[rb]:
            self.parent[ra] = rb
        elif self.rank[ra] > self.rank[rb]:
            self.parent[rb] = ra
        else:
            self.parent[rb] = ra
            self.rank[ra] += 1


def fof_group_loci(
    loci_df,
    radius_arcsec=3.0,
):
    """
    Transitive friends-of-friends grouping on the sky.

    A-B <= radius and B-C <= radius puts A/B/C in the
    same target even if A-C is slightly larger than radius.
    """
    work = (
        loci_df
        .dropna(subset=["ra", "dec"])
        .reset_index(drop=True)
        .copy()
    )

    xyz = radec_to_unit_xyz(
        work["ra"].values,
        work["dec"].values,
    )

    theta = np.deg2rad(
        radius_arcsec / 3600.0
    )

    # Euclidean chord distance on the unit sphere.
    chord_radius = 2.0 * np.sin(
        theta / 2.0
    )

    tree = cKDTree(xyz)

    pairs = tree.query_pairs(
        r=chord_radius,
        output_type="ndarray",
    )

    uf = UnionFind(len(work))

    for i, j in pairs:
        uf.union(
            int(i),
            int(j),
        )

    roots = np.array([
        uf.find(i)
        for i in range(len(work))
    ])

    unique_roots = {}

    group_index = []

    for root in roots:
        if root not in unique_roots:
            unique_roots[root] = len(unique_roots)

        group_index.append(
            unique_roots[root]
        )

    work["fof_group"] = group_index

    return work


## 3. Restrict alerts and select First-Cut loci

In [ ]:

alerts = alerts.copy()

alerts["mjd"] = pd.to_numeric(
    alerts["mjd"],
    errors="coerce",
)

n_alerts_original = len(alerts)

alerts = alerts[
    alerts["mjd"] >= FILTER_START_MJD
].copy()

n_after_mjd = len(alerts)

# Match the original Second-Pass application:
# only alerts containing lsst_diaSource_band enter feature construction.
has_band = alerts["alert_properties"].map(
    lambda p: has_prop(
        p,
        "lsst_diaSource_band",
    )
)

alerts = alerts[
    has_band
].copy()


print(
    f"Alerts before checks:            "
    f"{n_alerts_original:,}"
)

print(
    f"Alerts on/after MJD "
    f"{FILTER_START_MJD}: "
    f"{n_after_mjd:,}"
)

print(
    "Alerts with lsst_diaSource_band: "
    f"{len(alerts):,}"
)

if len(alerts):
    print(
        "Earliest retained MJD:          "
        f"{alerts['mjd'].min():.6f}"
    )


first_cut = loci.copy()

first_cut["first_cut_max_score"] = pd.to_numeric(
    first_cut["max_score"],
    errors="coerce",
)

first_cut["n_tagged_locus"] = pd.to_numeric(
    first_cut["num_tagged_alerts"],
    errors="coerce",
)

if first_cut["n_tagged_locus"].isna().any():
    raise ValueError(
        "Some downloaded loci have missing num_tagged_alerts."
    )

first_cut["n_tagged_locus"] = (
    first_cut["n_tagged_locus"]
    .astype(int)
)

first_cut = first_cut[
    first_cut["first_cut_max_score"]
    > FIRST_CUT_THRESHOLD
].copy()


print()
print(
    f"First-Cut loci: "
    f"{len(first_cut):,}"
)

print(
    "Minimum n_tagged among First-Cut loci:",
    first_cut["n_tagged_locus"].min(),
)


# The download query used num_tagged_alerts > 2.
if len(first_cut) and (
    first_cut["n_tagged_locus"] < 3
).any():
    raise ValueError(
        "Found a downloaded First-Cut locus with n_tagged < 3. "
        "This is inconsistent with the download query."
    )



## 4. Group First-Cut loci with 3″ friends-of-friends

Grouping is performed **before Second-Pass scoring**.

For a grouped target:

- all member-locus alerts are combined;
- `first_cut_max_score` = maximum First-Cut score across member loci;
- `n_tagged` = **sum** of the Lantern counts across member loci;
- target RA/Dec = spherical mean of the member-locus coordinates.

The `n_tagged` sum is the multi-locus rule we previously adopted: once nearby loci are treated as one target and their alert histories are combined, their Lantern tagged-alert counts are combined as well.


In [ ]:

grouped_loci = fof_group_loci(
    first_cut,
    radius_arcsec=FOF_RADIUS_ARCSEC,
)


target_rows = []
member_rows = []


for group_id, g in grouped_loci.groupby(
    "fof_group",
    sort=True,
):
    member_ids = sorted(
        g["locus_id"].astype(str).tolist()
    )

    representative_locus_id = member_ids[0]

    target_id = (
        f"target_{int(group_id):07d}"
    )

    target_ra, target_dec = spherical_mean_deg(
        g["ra"].values,
        g["dec"].values,
    )

    target_rows.append({
        "target_id": target_id,
        "representative_locus_id":
            representative_locus_id,
        "member_locus_ids": member_ids,
        "n_member_loci": len(member_ids),
        "ra": target_ra,
        "dec": target_dec,
        "first_cut_max_score":
            g["first_cut_max_score"].max(),
        # IMPORTANT: combine Lantern counts across member loci.
        "n_tagged":
            int(g["n_tagged_locus"].sum()),
    })

    for locus_id in member_ids:
        member_rows.append({
            "target_id": target_id,
            "locus_id": locus_id,
        })


target_meta = pd.DataFrame(
    target_rows
)

target_members = pd.DataFrame(
    member_rows
)


print(
    f"First-Cut loci:          "
    f"{len(first_cut):,}"
)

print(
    f"FoF targets:             "
    f"{len(target_meta):,}"
)

print(
    "Multi-locus targets:     "
    f"{(target_meta['n_member_loci'] > 1).sum():,}"
)

print(
    "Loci in multi-locus targets: "
    f"{target_meta.loc[target_meta['n_member_loci'] > 1, 'n_member_loci'].sum():,}"
)


# Every First-Cut locus must belong to exactly one FoF target.
if target_members["locus_id"].duplicated().any():
    raise RuntimeError(
        "A locus was assigned to more than one FoF target."
    )

if set(target_members["locus_id"]) != set(first_cut["locus_id"]):
    raise RuntimeError(
        "FoF membership does not contain exactly the First-Cut loci."
    )


# Verify the stored target-level n_tagged values against a fresh group sum.
check_n_tagged = (
    target_members
    .merge(
        first_cut[
            [
                "locus_id",
                "n_tagged_locus",
            ]
        ],
        on="locus_id",
        how="left",
    )
    .groupby("target_id")["n_tagged_locus"]
    .sum()
)

stored_n_tagged = (
    target_meta
    .set_index("target_id")["n_tagged"]
)

if not check_n_tagged.equals(
    stored_n_tagged.loc[check_n_tagged.index]
):
    raise RuntimeError(
        "Target-level n_tagged does not equal the sum over member loci."
    )


display(
    target_meta[
        target_meta["n_member_loci"] > 1
    ].head(20)
)


## 5. Map alerts to grouped targets

In [ ]:

locus_to_target = (
    target_members
    .set_index("locus_id")["target_id"]
)

target_alerts = alerts[
    alerts["locus_id"].isin(
        locus_to_target.index
    )
].copy()

target_alerts["target_id"] = (
    target_alerts["locus_id"]
    .map(locus_to_target)
)


# The clean hybrid download should already have globally unique alert IDs.
# Stop rather than silently altering the data if this assumption is violated.
if target_alerts["alert_id"].duplicated().any():
    raise ValueError(
        "Duplicate alert IDs found after target mapping."
    )


alert_counts = (
    target_alerts
    .groupby("target_id")
    .size()
    .rename("n_lsst_alerts")
)

target_meta["n_lsst_alerts"] = (
    target_meta["target_id"]
    .map(alert_counts)
    .fillna(0)
    .astype(int)
)

target_meta["bad_counter_flag"] = (
    target_meta["n_tagged"]
    > target_meta["n_lsst_alerts"]
)

target_meta["no_alert_flag"] = (
    target_meta["n_lsst_alerts"] == 0
)


print(
    f"Mapped alerts:                  "
    f"{len(target_alerts):,}"
)

print(
    f"Targets with n_tagged > alerts: "
    f"{target_meta['bad_counter_flag'].sum():,}"
)

print(
    f"Targets with no usable alerts:  "
    f"{target_meta['no_alert_flag'].sum():,}"
)


# Counter mismatches are retained and scored.
# Only targets with no usable alerts cannot produce features.
eligible_targets = target_meta[
    ~target_meta["no_alert_flag"]
].copy()


if target_meta["bad_counter_flag"].any():
    display(
        target_meta.loc[
            target_meta["bad_counter_flag"],
            [
                "target_id",
                "representative_locus_id",
                "member_locus_ids",
                "n_member_loci",
                "n_tagged",
                "n_lsst_alerts",
                "bad_counter_flag",
            ],
        ].head(30)
    )


print(
    f"Targets eligible for scoring: "
    f"{len(eligible_targets):,}"
)


## 6. Select all targets (or optional test sample)

In [ ]:

if (
    SAMPLE_FRAC is None
    or SAMPLE_FRAC >= 1
):
    targets = eligible_targets.copy()

else:
    n = max(
        1,
        int(
            round(
                len(eligible_targets)
                * SAMPLE_FRAC
            )
        ),
    )

    targets = eligible_targets.sample(
        n=n,
        random_state=RANDOM_STATE,
    ).copy()


selected_target_ids = set(
    targets["target_id"]
)

selected_alerts = target_alerts[
    target_alerts["target_id"].isin(
        selected_target_ids
    )
].copy()


print(
    f"Targets used: "
    f"{len(targets):,}"
)

print(
    f"Alerts used:  "
    f"{len(selected_alerts):,}"
)

print(
    "Multi-locus targets used: "
    f"{(targets['n_member_loci'] > 1).sum():,}"
)


## 7. Build alert-level Second-Pass quantities

In [ ]:

def build_alert_features(alert_subset):
    p = alert_subset[
        "alert_properties"
    ]

    keys = {
        "band":
            "lsst_diaSource_band",
        "x":
            "lsst_diaSource_x",
        "y":
            "lsst_diaSource_y",
        "xErr":
            "lsst_diaSource_xErr",
        "yErr":
            "lsst_diaSource_yErr",
        "ra":
            "lsst_diaSource_ra",
        "dec":
            "lsst_diaSource_dec",
        "scienceFlux":
            "lsst_diaSource_scienceFlux",
        "psfFlux":
            "lsst_diaSource_psfFlux",
        "apFlux":
            "lsst_diaSource_apFlux",
        "extendedness":
            "lsst_diaSource_extendedness",
        "dipoleLength":
            "lsst_diaSource_dipoleLength",
        "ixx":
            "lsst_diaSource_ixx",
        "iyy":
            "lsst_diaSource_iyy",
        "ixxPSF":
            "lsst_diaSource_ixxPSF",
        "iyyPSF":
            "lsst_diaSource_iyyPSF",
    }

    out = alert_subset[
        [
            "target_id",
            "locus_id",
            "alert_id",
            "mjd",
        ]
    ].copy()

    for name, key in keys.items():
        out[name] = p.map(
            lambda v, k=key:
                get_prop(v, k)
        )

    out["band"] = normalize_band(
        out["band"]
    )

    numeric = [
        c for c in keys
        if c != "band"
    ]

    out[numeric] = out[
        numeric
    ].apply(
        pd.to_numeric,
        errors="coerce",
    )

    psf_trace = (
        out["ixxPSF"]
        + out["iyyPSF"]
    )

    src_trace = (
        out["ixx"]
        + out["iyy"]
    )

    out["moment_ext"] = np.where(
        psf_trace != 0,
        src_trace / psf_trace,
        np.nan,
    )

    out["template_flux"] = (
        out["scienceFlux"]
        - out["psfFlux"]
    )

    out["flux_ext"] = np.where(
        out["psfFlux"] != 0,
        out["apFlux"]
        / out["psfFlux"],
        np.nan,
    )

    out["x_y_err"] = np.sqrt(
        out["xErr"]**2
        + out["yErr"]**2
    )

    return out


df = build_alert_features(
    selected_alerts
)

print(
    f"Alert-level feature rows: "
    f"{len(df):,}"
)

df.head()



## 8. Build one Second-Pass feature row per FoF target

The feature construction is the same as the earlier local Second-Pass notebook, except that rows are grouped by `target_id` instead of individual `locus_id`.

The original model feature is:

```python
percent_tagged_raw = 100 * n_tagged / n_detections
```

For the ANTARES counter inconsistency, we keep `n_tagged` unchanged but use:

```python
percent_tagged = min(percent_tagged_raw, 100)
```

Only the capped `percent_tagged` is passed to the model. `percent_tagged_raw` is saved only for diagnostics.


In [ ]:

def calculate_stats(
    df,
    target_meta,
):
    grouped = df.groupby(
        "target_id"
    )

    summary = grouped.agg(
        # Original Second-Pass application counts.
        # n_detections is the number of non-null x measurements.
        n_detections=(
            "x",
            "count",
        ),
        n_lsst_alerts=(
            "alert_id",
            "size",
        ),

        # Global statistics
        x_std=(
            "x",
            "std",
        ),
        y_std=(
            "y",
            "std",
        ),
        mean_x_y_err=(
            "x_y_err",
            "mean",
        ),
        median_x_y_err=(
            "x_y_err",
            "median",
        ),
        moment_ext_mean=(
            "moment_ext",
            "mean",
        ),
        moment_ext_std=(
            "moment_ext",
            "std",
        ),
        extendedness_mean=(
            "extendedness",
            "mean",
        ),
        extendedness_std=(
            "extendedness",
            "std",
        ),
        dipoleLength_mean=(
            "dipoleLength",
            "mean",
        ),
        dipoleLength_std=(
            "dipoleLength",
            "std",
        ),

        # Alert coordinate, with target-coordinate fallback below.
        alert_ra=(
            "ra",
            "first",
        ),
        alert_dec=(
            "dec",
            "first",
        ),

        mjd_min=(
            "mjd",
            "min",
        ),
        mjd_max=(
            "mjd",
            "max",
        ),
    ).reset_index()

    summary["centroid_std"] = np.sqrt(
        summary["x_std"]**2
        + summary["y_std"]**2
    )

    summary["centroid_instability"] = np.where(
        summary["median_x_y_err"] > 0,
        (
            summary["centroid_std"]
            / summary["median_x_y_err"]
        ),
        0.0,
    )

    summary["centroid_std"] = (
        summary["centroid_std"]
        .fillna(0.0)
    )

    summary["centroid_instability"] = (
        summary["centroid_instability"]
        .fillna(0.0)
    )

    for band in "ugrizy":
        band_data = (
            df[
                df["band"] == band
            ]
            .groupby("target_id")
            .agg(
                **{
                    f"apFlux_mean_{band}":
                        ("apFlux", "mean"),
                    f"apFlux_std_{band}":
                        ("apFlux", "std"),
                    f"template_flux_mean_{band}":
                        ("template_flux", "mean"),
                    f"flux_ext_mean_{band}":
                        ("flux_ext", "mean"),
                    f"flux_ext_std_{band}":
                        ("flux_ext", "std"),
                }
            )
        )

        summary = summary.merge(
            band_data,
            left_on="target_id",
            right_index=True,
            how="left",
        )

    meta = target_meta[
        [
            "target_id",
            "representative_locus_id",
            "member_locus_ids",
            "n_member_loci",
            "ra",
            "dec",
            "first_cut_max_score",
            "n_tagged",
            "bad_counter_flag",
        ]
    ].rename(
        columns={
            "ra": "target_ra",
            "dec": "target_dec",
        }
    )

    summary = summary.merge(
        meta,
        on="target_id",
        how="left",
    )

    summary["ra"] = (
        summary["alert_ra"]
        .fillna(
            summary["target_ra"]
        )
    )

    summary["dec"] = (
        summary["alert_dec"]
        .fillna(
            summary["target_dec"]
        )
    )

    # Reproduce the original model feature first.
    raw_percent = np.where(
        summary["n_detections"] > 0,
        (
            100.0
            * summary["n_tagged"]
            / summary["n_detections"]
        ),
        np.nan,
    )

    summary["percent_tagged_raw"] = np.round(
        raw_percent,
        5,
    )

    # Requested handling of ANTARES counter inconsistencies:
    # keep n_tagged, but never let the model see >100%.
    summary["percent_tagged"] = np.round(
        np.minimum(
            raw_percent,
            100.0,
        ),
        5,
    )

    summary["percent_tagged_capped_flag"] = (
        summary["percent_tagged_raw"]
        > 100.0
    )

    return summary


summary = calculate_stats(
    df,
    targets,
)


count_mismatch = (
    summary["n_detections"]
    != summary["n_lsst_alerts"]
)

counter_capped = (
    summary["percent_tagged_capped_flag"]
)


print(
    f"Second-Pass target rows: "
    f"{len(summary):,}"
)

print(
    "Targets with n_detections != n_lsst_alerts: "
    f"{count_mismatch.sum():,}"
)

print(
    "Targets with percent_tagged capped at 100%: "
    f"{counter_capped.sum():,}"
)


if counter_capped.any():
    display(
        summary.loc[
            counter_capped,
            [
                "target_id",
                "representative_locus_id",
                "member_locus_ids",
                "n_tagged",
                "n_detections",
                "n_lsst_alerts",
                "percent_tagged_raw",
                "percent_tagged",
                "bad_counter_flag",
            ],
        ].head(30)
    )


summary.head()


## 9. Validate grouped targets and Second-Pass features

In [ ]:

# Structural checks
if not summary["target_id"].is_unique:
    raise RuntimeError(
        "Second-Pass summary has duplicate target_id rows."
    )

if not selected_alerts["alert_id"].is_unique:
    raise RuntimeError(
        "Selected alerts are not unique by alert_id."
    )

if (
    summary["percent_tagged"]
    .dropna()
    .gt(100.0)
    .any()
):
    raise RuntimeError(
        "percent_tagged exceeds 100%."
    )

if (
    summary["percent_tagged"]
    .dropna()
    .lt(0.0)
    .any()
):
    raise RuntimeError(
        "percent_tagged is negative."
    )


# Verify that the 44 default model features are still exactly the
# established Second-Pass feature set.
expected_feature_count = 44

print(
    "Validated targets:              ",
    f"{len(summary):,}",
)

print(
    "Multi-locus targets:            ",
    f"{(summary['n_member_loci'] > 1).sum():,}",
)

print(
    "Counter-mismatch targets:       ",
    f"{summary['bad_counter_flag'].sum():,}",
)

print(
    "percent_tagged capped targets:  ",
    f"{summary['percent_tagged_capped_flag'].sum():,}",
)

print(
    "Maximum percent_tagged:         ",
    summary["percent_tagged"].max(),
)


## 10. Apply the Second-Pass model

In [ ]:

# Exact feature set/order from the current Second-Pass training notebook.
DEFAULT_MODEL_FEATURES = [
    "x_std",
    "y_std",
    "mean_x_y_err",
    "median_x_y_err",
    "moment_ext_mean",
    "moment_ext_std",
    "extendedness_mean",
    "extendedness_std",
    "dipoleLength_mean",
    "dipoleLength_std",
    "centroid_std",
    "centroid_instability",

    "apFlux_mean_u",
    "apFlux_std_u",
    "template_flux_mean_u",
    "flux_ext_mean_u",
    "flux_ext_std_u",

    "apFlux_mean_g",
    "apFlux_std_g",
    "template_flux_mean_g",
    "flux_ext_mean_g",
    "flux_ext_std_g",

    "apFlux_mean_r",
    "apFlux_std_r",
    "template_flux_mean_r",
    "flux_ext_mean_r",
    "flux_ext_std_r",

    "apFlux_mean_i",
    "apFlux_std_i",
    "template_flux_mean_i",
    "flux_ext_mean_i",
    "flux_ext_std_i",

    "apFlux_mean_z",
    "apFlux_std_z",
    "template_flux_mean_z",
    "flux_ext_mean_z",
    "flux_ext_std_z",

    "apFlux_mean_y",
    "apFlux_std_y",
    "template_flux_mean_y",
    "flux_ext_mean_y",
    "flux_ext_std_y",

    "n_tagged",
    "percent_tagged",
]


def get_model_features(model):
    if hasattr(
        model,
        "feature_names_in_",
    ):
        return list(
            model.feature_names_in_
        )

    try:
        names = (
            model
            .get_booster()
            .feature_names
        )

        if names:
            return list(names)

    except Exception:
        pass

    return DEFAULT_MODEL_FEATURES


model_features = get_model_features(
    model
)

if len(DEFAULT_MODEL_FEATURES) != 44:
    raise RuntimeError(
        f"DEFAULT_MODEL_FEATURES has {len(DEFAULT_MODEL_FEATURES)} features, expected 44."
    )

if len(model_features) != 44:
    raise RuntimeError(
        f"The loaded model expects {len(model_features)} features, expected 44."
    )



missing = [
    c
    for c in model_features
    if c not in summary.columns
]

if missing:
    raise KeyError(
        "Model requires features not constructed here:\n"
        + "\n".join(missing)
    )


X = summary[
    model_features
].replace(
    [np.inf, -np.inf],
    np.nan,
)


print(
    "Model input features:",
    len(model_features),
)


summary["second_pass_score"] = (
    model.predict_proba(X)[:, 1]
)

summary["candidate_flag"] = (
    summary["second_pass_score"]
    > SECOND_PASS_THRESHOLD
)


candidates = (
    summary[
        summary["candidate_flag"]
    ]
    .sort_values(
        "second_pass_score",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    f"Candidates: "
    f"{len(candidates):,} / "
    f"{len(summary):,} "
    f"(score > {SECOND_PASS_THRESHOLD})"
)

print(
    "Multi-locus candidates:",
    int(
        (
            candidates["n_member_loci"]
            > 1
        ).sum()
    ),
)


## 11. Candidate list

In [ ]:

candidate_columns = [
    "target_id",
    "representative_locus_id",
    "member_locus_ids",
    "n_member_loci",
    "ra",
    "dec",
    "first_cut_max_score",
    "n_tagged",
    "n_lsst_alerts",
    "n_detections",
    "percent_tagged",
    "percent_tagged_capped_flag",
    "bad_counter_flag",
    "mjd_min",
    "mjd_max",
    "second_pass_score",
]


candidates[
    candidate_columns
].head(50)


## 12. Save outputs

In [ ]:

candidate_target_ids = set(
    candidates["target_id"]
)

candidate_alerts = selected_alerts[
    selected_alerts["target_id"].isin(
        candidate_target_ids
    )
].copy()


# CSV-friendly copy: join the member locus IDs.
candidate_csv = candidates[
    candidate_columns
].copy()

candidate_csv["member_locus_ids"] = (
    candidate_csv[
        "member_locus_ids"
    ].map(
        lambda x: "|".join(x)
        if isinstance(
            x,
            (list, tuple, np.ndarray),
        )
        else str(x)
    )
)


# 1. Target membership map
target_members.to_parquet(
    OUTPUT_DIR
    / "second_pass_target_members.parquet",
    index=False,
)

# 2. All grouped target metadata
target_meta.to_parquet(
    OUTPUT_DIR
    / "second_pass_target_metadata.parquet",
    index=False,
)

# 3. Counter mismatches for QC only; these are NOT excluded.
target_meta[
    target_meta["bad_counter_flag"]
].to_csv(
    OUTPUT_DIR
    / "second_pass_counter_mismatches.csv",
    index=False,
)

# 4. Full score table
summary.to_parquet(
    OUTPUT_DIR
    / "second_pass_scored_targets.parquet",
    index=False,
)

# 5. Compact candidate lists
candidate_csv.to_csv(
    OUTPUT_DIR
    / "second_pass_candidates.csv",
    index=False,
)

candidates[
    candidate_columns
].to_parquet(
    OUTPUT_DIR
    / "second_pass_candidates.parquet",
    index=False,
)

# 6. All retained alerts belonging to candidate targets
candidate_alerts.to_parquet(
    OUTPUT_DIR
    / "second_pass_candidate_alerts.parquet",
    index=False,
)


print("Saved:")
print(
    "  second_pass_target_members.parquet"
)
print(
    "  second_pass_target_metadata.parquet"
)
print(
    "  second_pass_counter_mismatches.csv"
)
print(
    "  second_pass_scored_targets.parquet"
)
print(
    "  second_pass_candidates.csv"
)
print(
    "  second_pass_candidates.parquet"
)
print(
    "  second_pass_candidate_alerts.parquet"
)


## 13. Score distribution

In [ ]:

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

plt.figure(
    figsize=(4, 3)
)

plt.hist(
    summary["second_pass_score"],
    bins=30,
    histtype="step",
)

plt.axvline(
    SECOND_PASS_THRESHOLD,
    ls="--",
)

plt.xlabel(
    "Second-Pass score"
)

plt.ylabel(
    "Targets"
)

plt.tight_layout()
plt.show()



## Notes

- `SAMPLE_FRAC = 1.0` runs **all** grouped targets.
- FoF grouping uses a 3″ linking length and is transitive.
- All retained alerts from all member loci are combined before feature construction.
- The 44 model features are unchanged from the earlier local Second-Pass implementation.
- For a multi-locus target, `n_tagged` is the **sum** of `num_tagged_alerts` across member loci.
- `first_cut_max_score` is the maximum First-Cut score across member loci.
- `bad_counter_flag = (n_tagged > n_lsst_alerts)` is diagnostic only; these targets are still scored.
- The original model definition uses `n_detections` for `percent_tagged`.
- `percent_tagged_raw = 100 * n_tagged / n_detections` is saved only for diagnostics.
- `percent_tagged = min(percent_tagged_raw, 100)` is the value passed to the model.
- Targets with no usable post-cut LSST alerts are the only grouped targets excluded before feature construction.
- Duplicate loci or duplicate alert IDs now stop the notebook instead of being silently modified.
